In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# BACK_AUG front back-row 강화 합성 데이터 생성
# ============================================================
# - 기존 S0~S7과 별도 추가 데이터셋
# - scenario_code = BACK_AUG
# - 전체 선반 이미지 새로 생성
# - YOLO txt는 product_class_map.py 기준 class_id 사용
# - row_no == 1: front/back 모두 포함
# - row_no != 1: front 포함, 같은 lane에 front 없을 때만 back 포함
# ============================================================

import os
import sys
import json
import random
import shutil
from pathlib import Path
from collections import Counter
from datetime import datetime

from PIL import Image, ImageDraw
from IPython.display import display
from tqdm.auto import tqdm

BASE = Path("/content/drive/MyDrive/먼작귀")

# 코드 위치
SCENARIO_DIR = BASE / "synthetic_scenarios"
FRONT_SCENARIO_DIR = SCENARIO_DIR / "front"

# shelf_synthetic_common.py 위치
os.chdir(FRONT_SCENARIO_DIR)

if str(FRONT_SCENARIO_DIR) not in sys.path:
    sys.path.insert(0, str(FRONT_SCENARIO_DIR))

# product_class_map.py 위치
if str(SCENARIO_DIR) not in sys.path:
    sys.path.insert(0, str(SCENARIO_DIR))

import shelf_synthetic_common as common

from product_class_map import (
    get_product_class_id_from_obj,
    get_product_name_from_obj,
    PRODUCT_CLASS_NAMES,
    validate_class_map,
)

print("현재 위치:", Path.cwd())
print("common file:", common.__file__)
print("class map 검수:", validate_class_map())

# 결과 저장 위치
AUG_ROOT = BASE / "dataset" / "synthetic_front_back_aug"

AUG_IMAGE_ROOT = AUG_ROOT / "images" / "front" / "back_aug"
AUG_JSON_ROOT = AUG_ROOT / "labels_json" / "front" / "back_aug"
AUG_YOLO_ROOT = AUG_ROOT / "labels_yolo" / "front" / "back_aug"
AUG_SEG_ROOT = AUG_ROOT / "labels_yolo_seg" / "shelf_lip" / "front" / "back_aug"

print("AUG_ROOT:", AUG_ROOT)

현재 위치: /content/drive/.shortcut-targets-by-id/1qwubl3Zfy5buxG2NKhpBhCZdTiZ64Tx9/먼작귀/synthetic_scenarios/front
common file: /content/drive/MyDrive/먼작귀/synthetic_scenarios/front/shelf_synthetic_common.py
class map 검수: {'num_classes': 67, 'min_class_id': 0, 'max_class_id': 66, 'num_name_map': 67}
AUG_ROOT: /content/drive/MyDrive/먼작귀/dataset/synthetic_front_back_aug


In [ ]:
# ============================================================
# 1. 생성 개수 / 케이스 비율
# ============================================================

AUG_TOTAL_COUNT = 600

CASE_WEIGHTS = {
    "top_front_back": 0.35,           # 맨 위층 front + back 둘 다 있음
    "top_back_only_lane": 0.25,       # 맨 위층 일부 lane은 front 없음 + back 있음
    "middle_front_empty_back": 0.25,  # 중간/아래층 front 없음 + back 보임
    "dense_back_row": 0.15,           # 촘촘한 back-row
}

TOP_ROW_NO = 1

print("총 생성 수:", AUG_TOTAL_COUNT)
print("case 비율:", CASE_WEIGHTS)

총 생성 수: 600
case 비율: {'top_front_back': 0.35, 'top_back_only_lane': 0.25, 'middle_front_empty_back': 0.25, 'dense_back_row': 0.15}


In [ ]:
# ============================================================
# 2. BACK_AUG slot_plan 생성 함수
# ============================================================

def choose_case_type(rng):
    case_names = list(CASE_WEIGHTS.keys())
    weights = list(CASE_WEIGHTS.values())
    return rng.choices(case_names, weights=weights, k=1)[0]


def random_indices(qty, rng, min_k=1, max_k=None):
    if qty <= 0:
        return []

    if max_k is None:
        max_k = qty

    max_k = min(max_k, qty)
    min_k = min(min_k, max_k)

    k = rng.randint(min_k, max_k)
    return sorted(rng.sample(list(range(qty)), k=k))


def get_top_slot_ids(plan):
    return [
        slot_id for slot_id, p in plan.items()
        if int(p["slot"]["row_no"]) == TOP_ROW_NO
    ]


def get_non_top_slot_ids(plan):
    return [
        slot_id for slot_id, p in plan.items()
        if int(p["slot"]["row_no"]) != TOP_ROW_NO
    ]


def set_common_slot_metadata(p, back_aug_case_type, action, list_up=False, missing_qty=0):
    """
    BACK_AUG 전용 메타데이터.
    기존 S0~S7과 분리해서 관리.
    """
    p["scenario_code"] = "BACK_AUG"
    p["sub_scenario_code"] = back_aug_case_type
    p["back_aug_case_type"] = back_aug_case_type
    p["action"] = action
    p["final_status"] = "back_row_augmented"
    p["list_up"] = list_up
    p["missing_qty"] = missing_qty
    p["is_misplaced"] = False
    return p


def apply_top_front_back_case(plan, rng):
    """
    맨 위층 row_no == 1에서 front + back 둘 다 있는 케이스.
    """
    top_slots = get_top_slot_ids(plan)

    selected_slots = rng.sample(
        top_slots,
        k=min(len(top_slots), rng.randint(3, 6))
    )

    for slot_id in selected_slots:
        p = plan[slot_id]
        normal_qty = int(p["normal_front_qty"])

        p["display_qty"] = normal_qty
        p["back_display_qty"] = normal_qty
        p["front_missing_indices"] = []
        p["back_missing_indices"] = []
        p["back_visible_indices"] = None

        set_common_slot_metadata(
            p=p,
            back_aug_case_type="top_front_back",
            action="top_front_back",
            list_up=False,
            missing_qty=0,
        )

    return plan


def apply_top_back_only_lane_case(plan, rng):
    """
    맨 위층에서 일부 lane은 front 없이 back만 있는 케이스.
    """
    top_slots = get_top_slot_ids(plan)

    selected_slots = rng.sample(
        top_slots,
        k=min(len(top_slots), rng.randint(3, 6))
    )

    for slot_id in selected_slots:
        p = plan[slot_id]
        normal_qty = int(p["normal_front_qty"])

        missing_front = random_indices(
            qty=normal_qty,
            rng=rng,
            min_k=1,
            max_k=max(1, normal_qty // 2)
        )

        p["display_qty"] = normal_qty
        p["back_display_qty"] = normal_qty

        # 앞줄 일부 제거, 뒷줄은 유지
        p["front_missing_indices"] = missing_front
        p["back_missing_indices"] = []
        p["back_visible_indices"] = None

        set_common_slot_metadata(
            p=p,
            back_aug_case_type="top_back_only_lane",
            action="top_back_only_lane",
            list_up=True,
            missing_qty=len(missing_front),
        )

    return plan


def apply_middle_front_empty_back_case(plan, rng):
    """
    중간/아래층에서 일부 lane의 front를 비워서 back이 보이는 케이스.
    """
    non_top_slots = get_non_top_slot_ids(plan)

    selected_slots = rng.sample(
        non_top_slots,
        k=min(len(non_top_slots), rng.randint(4, 8))
    )

    for slot_id in selected_slots:
        p = plan[slot_id]
        normal_qty = int(p["normal_front_qty"])

        front_empty_indices = random_indices(
            qty=normal_qty,
            rng=rng,
            min_k=1,
            max_k=max(1, normal_qty // 2)
        )

        p["display_qty"] = normal_qty
        p["back_display_qty"] = normal_qty

        # front가 빠진 위치에서만 back을 보이게 생성
        p["front_missing_indices"] = front_empty_indices
        p["back_missing_indices"] = []
        p["back_visible_indices"] = front_empty_indices

        set_common_slot_metadata(
            p=p,
            back_aug_case_type="middle_front_empty_back",
            action="middle_front_empty_back",
            list_up=True,
            missing_qty=len(front_empty_indices),
        )

    return plan


def apply_dense_back_row_case(plan, rng):
    """
    같은 상품이 촘촘히 붙어 있는 front back-row 케이스.
    주로 맨 위층에 적용.
    """
    top_slots = get_top_slot_ids(plan)

    selected_slots = rng.sample(
        top_slots,
        k=min(len(top_slots), rng.randint(3, 6))
    )

    for slot_id in selected_slots:
        p = plan[slot_id]
        normal_qty = int(p["normal_front_qty"])

        dense_qty = normal_qty + rng.randint(1, 2)

        p["display_qty"] = dense_qty
        p["back_display_qty"] = dense_qty
        p["front_missing_indices"] = []
        p["back_missing_indices"] = []
        p["back_visible_indices"] = None

        set_common_slot_metadata(
            p=p,
            back_aug_case_type="dense_back_row",
            action="dense_back_row",
            list_up=False,
            missing_qty=0,
        )

    return plan


def make_front_back_aug_plan(ctx, case_type, rng):
    """
    BACK_AUG 전용 slot_plan 생성.
    """
    plan = common.make_normal_slot_plan(ctx)

    # 전체 슬롯 기본값 BACK_AUG로 통일
    for _, p in plan.items():
        p["scenario_code"] = "BACK_AUG"
        p["sub_scenario_code"] = "normal"
        p["back_aug_case_type"] = "none"
        p["action"] = "normal"
        p["final_status"] = "normal"
        p["list_up"] = False
        p["missing_qty"] = 0
        p["is_misplaced"] = False

    if case_type == "top_front_back":
        return apply_top_front_back_case(plan, rng)

    if case_type == "top_back_only_lane":
        return apply_top_back_only_lane_case(plan, rng)

    if case_type == "middle_front_empty_back":
        return apply_middle_front_empty_back_case(plan, rng)

    if case_type == "dense_back_row":
        return apply_dense_back_row_case(plan, rng)

    raise ValueError(f"알 수 없는 case_type: {case_type}")

In [ ]:
# ============================================================
# 3. YOLO 라벨 생성 규칙
# ============================================================
# 중요:
# - common.objects_to_yolo_lines() 사용 X
# - JSON 내부 class_id 사용 X
# - product_class_map.py 기준 product_name → class_id 매핑
# ============================================================

def get_depth_from_obj(obj):
    return obj.get("depth_row", obj.get("depth_gt", "front"))


def get_lane_key_from_obj(obj):
    return (
        obj.get("slot_id"),
        obj.get("position_index"),
    )


def select_objects_for_front_yolo(objects):
    """
    최종 YOLO 포함 규칙.

    row_no == 1:
        front 포함
        back 포함

    row_no != 1:
        front 포함
        같은 lane에 front 있으면 back 제외
        같은 lane에 front 없으면 back 포함
    """
    top_objects = []
    non_top_objects = []

    for obj in objects:
        row_no = int(obj.get("row_no"))
        depth = get_depth_from_obj(obj)

        if depth not in ["front", "back"]:
            continue

        if row_no == TOP_ROW_NO:
            top_objects.append(obj)
        else:
            non_top_objects.append(obj)

    selected = []

    # 맨 위층은 front/back 모두 포함
    selected.extend(top_objects)

    # 나머지 층은 lane 기준
    front_keys = set()

    for obj in non_top_objects:
        if get_depth_from_obj(obj) == "front":
            front_keys.add(get_lane_key_from_obj(obj))

    for obj in non_top_objects:
        depth = get_depth_from_obj(obj)
        key = get_lane_key_from_obj(obj)

        if depth == "front":
            selected.append(obj)

        elif depth == "back":
            if key not in front_keys:
                selected.append(obj)

    return selected


def object_to_class_id_by_product_map(obj):
    """
    product_class_map.py 기준으로 class_id 매핑.
    JSON 내부 class_id는 절대 사용하지 않음.
    """
    class_id = get_product_class_id_from_obj(obj, default=None)

    if class_id is None:
        product_name = get_product_name_from_obj(obj)
        raise KeyError(f"product_class_map 매칭 실패: {product_name}")

    return int(class_id)


def selected_objects_to_yolo_lines(image, selected_objects):
    image_w, image_h = image.size
    lines = []

    for obj in selected_objects:
        class_id = object_to_class_id_by_product_map(obj)

        x1, y1, x2, y2 = obj["bbox"]

        x1 = max(0, min(float(x1), image_w))
        x2 = max(0, min(float(x2), image_w))
        y1 = max(0, min(float(y1), image_h))
        y2 = max(0, min(float(y2), image_h))

        box_w = x2 - x1
        box_h = y2 - y1

        if box_w <= 0 or box_h <= 0:
            continue

        xc = ((x1 + x2) / 2) / image_w
        yc = ((y1 + y2) / 2) / image_h
        w = box_w / image_w
        h = box_h / image_h

        lines.append(f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

    return lines

In [ ]:
# ============================================================
# 4. JSON / shelf_lip 변환 함수
# ============================================================

def xyxy_to_yolo_bbox(bbox, image_w, image_h):
    x1, y1, x2, y2 = bbox

    x1 = max(0, min(float(x1), image_w))
    x2 = max(0, min(float(x2), image_w))
    y1 = max(0, min(float(y1), image_h))
    y2 = max(0, min(float(y2), image_h))

    box_w = x2 - x1
    box_h = y2 - y1

    if box_w <= 0 or box_h <= 0:
        return None

    xc = ((x1 + x2) / 2) / image_w
    yc = ((y1 + y2) / 2) / image_h
    w = box_w / image_w
    h = box_h / image_h

    return [round(xc, 6), round(yc, 6), round(w, 6), round(h, 6)]


def points_to_yolo(points, image_w, image_h):
    return [
        [round(x / image_w, 6), round(y / image_h, 6)]
        for x, y in points
    ]


def convert_objects_for_json(objects, selected_objects, image_w, image_h):
    selected_keys = {
        (
            obj.get("object_id"),
            obj.get("slot_id"),
            obj.get("position_index"),
            get_depth_from_obj(obj),
        )
        for obj in selected_objects
    }

    json_objects = []

    for obj in objects:
        new_obj = dict(obj)

        depth_gt = get_depth_from_obj(obj)
        bbox = obj["bbox"]
        x1, y1, x2, y2 = bbox

        class_id = object_to_class_id_by_product_map(obj)
        bbox_yolo = xyxy_to_yolo_bbox(bbox, image_w, image_h)

        key = (
            obj.get("object_id"),
            obj.get("slot_id"),
            obj.get("position_index"),
            depth_gt,
        )

        new_obj["class_id"] = class_id
        new_obj["depth_gt"] = depth_gt
        new_obj["bbox_xyxy_full"] = [int(x1), int(y1), int(x2), int(y2)]
        new_obj["bbox_yolo_full"] = bbox_yolo
        new_obj["bbox_center_xy"] = [int((x1 + x2) / 2), int((y1 + y2) / 2)]
        new_obj["bbox_bottom_center_xy"] = [int((x1 + x2) / 2), int(y2)]
        new_obj["visible"] = True
        new_obj["included_in_product_yolo"] = key in selected_keys

        json_objects.append(new_obj)

    return json_objects


def convert_slots_for_json(ctx, slot_labels):
    slot_map = {slot["slot_id"]: slot for slot in ctx["all_slots"]}
    json_slots = []

    for s in slot_labels:
        slot_id = s["slot_id"]
        slot = slot_map[slot_id]

        front_display_qty = int(s.get("front_display_qty", 0))
        back_display_qty = int(s.get("back_display_qty", 0))
        required_front_qty = int(s.get("normal_front_qty") or 0)
        required_back_qty = int(s.get("normal_back_qty") or 0)

        front_missing_qty = max(0, required_front_qty - front_display_qty)
        back_missing_qty = max(0, required_back_qty - back_display_qty)
        missing_qty = int(s.get("missing_qty") or front_missing_qty)

        is_misplaced = bool(s.get("is_misplaced", False))
        is_slot_empty = (front_display_qty + back_display_qty) == 0
        is_front_depleted = front_display_qty == 0 and required_front_qty > 0

        store_stock_qty = 25
        reorder_point = 5

        if is_misplaced:
            status_code = "CHECK_REQUIRED"
            status_label = "확인 필요"
        elif missing_qty <= 0:
            status_code = "NORMAL"
            status_label = "정상"
        elif store_stock_qty <= reorder_point:
            status_code = "ORDER_REQUIRED"
            status_label = "발주 필요"
        else:
            status_code = "REPLENISH_REQUIRED"
            status_label = "보충 필요"

        json_slots.append({
            "slot_id": slot_id,
            "shelf_id": "SHELF_A",
            "shelf_no": 1,
            "row_no": int(slot["row_no"]),
            "col_no": int(slot["col_no"]),
            "location_label": f"선반 A {slot['row_no']}-{slot['col_no']}",

            "slot_geometry": {
                "slot_bbox_xyxy": [slot["x1"], slot["y1"], slot["x2"], slot["y2"]],
                "slot_polygon_xy": [
                    [slot["x1"], slot["y1"]],
                    [slot["x2"], slot["y1"]],
                    [slot["x2"], slot["y2"]],
                    [slot["x1"], slot["y2"]],
                ],
            },

            "scenario_code": s.get("scenario_code", "BACK_AUG"),
            "sub_scenario_code": s.get("sub_scenario_code", "none"),
            "back_aug_case_type": s.get("back_aug_case_type", "none"),
            "action": s.get("action"),

            "expected_product_id": s.get("target_product_id"),
            "expected_product_name": s.get("target_product_name"),
            "actual_product_id": s.get("actual_product_id"),
            "actual_product_name": s.get("actual_product_name"),

            "required_front_qty": required_front_qty,
            "required_back_qty": required_back_qty,
            "front_display_qty": front_display_qty,
            "back_display_qty": back_display_qty,

            "front_missing_qty": front_missing_qty,
            "back_missing_qty": back_missing_qty,
            "missing_qty": missing_qty,
            "missing_ratio": round(
                missing_qty / max(1, required_front_qty + required_back_qty),
                4
            ),

            "front_missing_indices": s.get("front_missing_indices", []),
            "back_missing_indices": s.get("back_missing_indices", []),
            "target_column_index": s.get("target_column_index"),

            "is_misplaced": is_misplaced,
            "is_front_depleted": is_front_depleted,
            "is_slot_empty": is_slot_empty,

            "store_stock_qty": store_stock_qty,
            "reorder_point": reorder_point,
            "recommended_replenish_qty": max(0, front_missing_qty),
            "recommended_order_qty": 0 if store_stock_qty > reorder_point else max(0, front_missing_qty),

            "status_code": status_code,
            "status_label": status_label,
            "list_up": bool(s.get("list_up", False)),
            "priority_score": 80 if status_code != "NORMAL" else 0,
            "priority_label": "high" if status_code != "NORMAL" else "none",

            "detected_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
        })

    return json_slots


def make_front_lines(ctx):
    lines = []

    row_points = common.ROW_POINTS
    points_per_row = common.POINTS_PER_ROW
    num_rows = len(row_points) // points_per_row

    for row_idx in range(num_rows):
        row_no = row_idx + 1
        start = row_idx * points_per_row
        row_pts = row_points[start:start + points_per_row]

        left = row_pts[0]
        right = row_pts[-1]

        lines.append({
            "line_id": f"row_{row_no}_front_line",
            "row_no": row_no,
            "points_xy": [[left[0], left[1]], [right[0], right[1]]],
            "source": "row_points",
            "front_band_px": 80,
            "back_band_px": 160,
        })

    return lines


def make_shelf_lips(ctx, image_w, image_h):
    shelf_lips = []

    for idx, poly in enumerate(ctx["shelf_lip_polygons"], start=1):
        shelf_lips.append({
            "shelf_lip_id": f"shelf_lip_{idx}",
            "shelf_no": idx,
            "points_xy": [[int(x), int(y)] for x, y in poly],
            "points_yolo": points_to_yolo(poly, image_w, image_h),
            "included_in_shelf_lip_yolo_seg": True,
        })

    return shelf_lips


def make_shelf_lip_yolo_seg_lines(ctx, image_w, image_h):
    lines = []

    for poly in ctx["shelf_lip_polygons"]:
        pts = []
        for x, y in poly:
            pts.append(f"{x / image_w:.6f}")
            pts.append(f"{y / image_h:.6f}")

        lines.append("0 " + " ".join(pts))

    return lines


def make_overview(json_slots):
    counter = Counter(s["status_code"] for s in json_slots)

    return {
        "total_slot_count": len(json_slots),
        "normal_slot_count": counter.get("NORMAL", 0),
        "replenish_required_count": counter.get("REPLENISH_REQUIRED", 0),
        "order_required_count": counter.get("ORDER_REQUIRED", 0),
        "check_required_count": counter.get("CHECK_REQUIRED", 0),
        "list_up_count": sum(1 for s in json_slots if s.get("list_up")),
        "misplaced_slot_count": sum(1 for s in json_slots if s.get("is_misplaced")),
        "empty_slot_count": sum(1 for s in json_slots if s.get("is_slot_empty")),
        "front_depleted_slot_count": sum(1 for s in json_slots if s.get("is_front_depleted")),
    }


def make_work_list(json_slots):
    work_items = [
        s for s in json_slots
        if s["status_code"] != "NORMAL"
    ]

    work_items = sorted(
        work_items,
        key=lambda x: x.get("priority_score", 0),
        reverse=True
    )

    result = []

    for rank, s in enumerate(work_items, start=1):
        result.append({
            "rank": rank,
            "slot_id": s["slot_id"],
            "product_id": s["actual_product_id"],
            "product_name": s["actual_product_name"],
            "status_code": s["status_code"],
            "status_label": s["status_label"],
            "store_stock_qty": s["store_stock_qty"],
            "recommended_replenish_qty": s["recommended_replenish_qty"],
            "recommended_order_qty": s["recommended_order_qty"],
            "location_label": s["location_label"],
            "detected_at": s["detected_at"],
            "priority_score": s["priority_score"],
            "priority_label": s["priority_label"],
        })

    return result


def make_sku_detail_candidates(json_slots):
    candidates = []

    for s in json_slots:
        if s["status_code"] == "NORMAL":
            continue

        candidates.append({
            "slot_id": s["slot_id"],
            "sku_id": s["actual_product_id"],
            "sku_name": s["actual_product_name"],
            "current_status": s["status_label"],
            "detected_at": s["detected_at"],
            "store_stock_qty": s["store_stock_qty"],
            "recommended_replenish_qty": s["recommended_replenish_qty"],
            "recommended_order_qty": s["recommended_order_qty"],
            "location_label": s["location_label"],
            "confidence": {
                "source": "synthetic_demo",
                "value": 0.65,
            }
        })

    return candidates


def make_output_json(
    ctx,
    image_id,
    file_name,
    image_path,
    case_type,
    seed,
    image_w,
    image_h,
    objects,
    selected_objects,
    slot_labels,
):
    json_objects = convert_objects_for_json(
        objects=objects,
        selected_objects=selected_objects,
        image_w=image_w,
        image_h=image_h,
    )

    json_slots = convert_slots_for_json(ctx, slot_labels)

    return {
        "image": {
            "image_id": image_id,
            "file_name": file_name,
            "image_path": str(image_path),
            "image_type": "SIMULATION_FRONT_BACK_AUG",
            "view": "front",
            "camera_id": "CAM_FRONT_01",
            "store_id": "STORE_001",
            "shelf_id": "SHELF_A",

            "scenario_code": "BACK_AUG",
            "scenario_name": "정면 뒤줄 탐지 강화",
            "sub_scenario_code": case_type,
            "back_aug_case_type": case_type,

            "created_seed": seed,
            "captured_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
            "image_width": image_w,
            "image_height": image_h,
            "background_name": "선반이미지_정면.png",
        },

        "overview": make_overview(json_slots),

        "planogram": {
            "planogram_id": "FRONT_SHELF_A_BACK_AUG_V1",
            "shelf_id": "SHELF_A",
            "view": "front",
            "slots": [
                {
                    "slot_id": slot["slot_id"],
                    "row_no": int(slot["row_no"]),
                    "col_no": int(slot["col_no"]),
                    "zone_id": slot["zone_id"],
                    "category": slot["category"],
                }
                for slot in ctx["all_slots"]
            ],
        },

        "front_lines": make_front_lines(ctx),
        "slots": json_slots,
        "objects": json_objects,
        "shelf_lips": make_shelf_lips(ctx, image_w, image_h),
        "work_list": make_work_list(json_slots),
        "sku_detail_candidates": make_sku_detail_candidates(json_slots),

        "class_map": {
            str(class_id): name
            for class_id, name in PRODUCT_CLASS_NAMES.items()
        },

        "settings": {
            "generator_version": "front_back_row_aug_v1",
            "modeling_strategy": "full_image_yolo_or_sahi_slicing_inference",
            "coordinate_system": "full_image",

            "augmentation_type": "front_back_row_structural_generation",
            "back_aug_case_type": case_type,

            "product_yolo_include_rule": {
                "top_row": "row_no == 1 includes both front and back objects",
                "non_top_row": "include front; include back only when same lane has no front",
                "empty_position": "no object, no label",
            },

            "product_class_rule": "JSON internal class_id is not used. product_name is remapped using product_class_map.py.",

            "shelf_lip_class": {
                "0": "shelf_lip",
            },

            "sahi_config": {
                "enabled": True,
                "slice_height": 640,
                "slice_width": 640,
                "overlap_height_ratio": 0.20,
                "overlap_width_ratio": 0.20,
                "postprocess_type": "NMS",
            },

            "status_rule": {
                "NORMAL": "no issue",
                "REPLENISH_REQUIRED": "missing exists and store stock is greater than reorder point",
                "ORDER_REQUIRED": "missing exists and store stock is less than or equal to reorder point",
                "CHECK_REQUIRED": "misplaced or abnormal display",
            },
        },
    }

In [ ]:
# ============================================================
# 5. 실제 생성 실행
# ============================================================

RESET_AUG_ROOT = True

if RESET_AUG_ROOT and AUG_ROOT.exists():
    shutil.rmtree(AUG_ROOT)
    print("기존 증강 폴더 삭제:", AUG_ROOT)

AUG_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
AUG_JSON_ROOT.mkdir(parents=True, exist_ok=True)
AUG_YOLO_ROOT.mkdir(parents=True, exist_ok=True)
AUG_SEG_ROOT.mkdir(parents=True, exist_ok=True)

rng = random.Random(20260525)

created_logs = []
case_counter = Counter()
error_logs = []

print(f"BACK_AUG 생성 시작: {AUG_TOTAL_COUNT}장")

for i in tqdm(range(1, AUG_TOTAL_COUNT + 1), desc="generate BACK_AUG"):
    try:
        case_type = choose_case_type(rng)
        render_seed = rng.randint(0, 999999)

        # 이미지마다 context seed를 바꿔 상품 배치 다양화
        ctx = common.create_synthetic_context(
            base_dir=BASE,
            background_name="선반이미지_정면.png",
            seed=render_seed,
        )

        slot_plan = make_front_back_aug_plan(
            ctx=ctx,
            case_type=case_type,
            rng=rng,
        )

        image, objects, slot_labels = common.render_from_slot_plan(
            ctx=ctx,
            slot_plan=slot_plan,
            seed=render_seed,
        )

        selected_objects = select_objects_for_front_yolo(objects)
        yolo_lines = selected_objects_to_yolo_lines(image, selected_objects)

        image_w, image_h = image.size

        stem = f"front_back_aug_{i:04d}_{case_type}"
        file_name = f"{stem}.png"

        out_img_path = AUG_IMAGE_ROOT / file_name
        out_json_path = AUG_JSON_ROOT / f"{stem}.json"
        out_yolo_path = AUG_YOLO_ROOT / f"{stem}.txt"
        out_seg_path = AUG_SEG_ROOT / f"{stem}.txt"

        # 이미지 저장
        image.convert("RGB").save(out_img_path)

        # 상품 YOLO txt 저장
        out_yolo_path.write_text(
            "\n".join(yolo_lines) + ("\n" if yolo_lines else ""),
            encoding="utf-8"
        )

        # shelf_lip YOLO-seg 저장
        seg_lines = make_shelf_lip_yolo_seg_lines(ctx, image_w, image_h)
        out_seg_path.write_text(
            "\n".join(seg_lines) + ("\n" if seg_lines else ""),
            encoding="utf-8"
        )

        # JSON 저장
        json_data = make_output_json(
            ctx=ctx,
            image_id=stem,
            file_name=file_name,
            image_path=out_img_path,
            case_type=case_type,
            seed=render_seed,
            image_w=image_w,
            image_h=image_h,
            objects=objects,
            selected_objects=selected_objects,
            slot_labels=slot_labels,
        )

        with open(out_json_path, "w", encoding="utf-8") as f:
            json.dump(json_data, f, ensure_ascii=False, indent=2)

        case_counter[case_type] += 1

        created_logs.append({
            "scenario_code": "BACK_AUG",
            "case_type": case_type,
            "stem": stem,
            "image": str(out_img_path),
            "json": str(out_json_path),
            "yolo": str(out_yolo_path),
            "seg": str(out_seg_path),
            "objects": len(objects),
            "selected_yolo": len(yolo_lines),
            "top_back_selected": sum(
                1 for obj in selected_objects
                if int(obj.get("row_no")) == TOP_ROW_NO and get_depth_from_obj(obj) == "back"
            ),
            "non_top_back_selected": sum(
                1 for obj in selected_objects
                if int(obj.get("row_no")) != TOP_ROW_NO and get_depth_from_obj(obj) == "back"
            ),
        })

    except Exception as e:
        error_logs.append({
            "index": i,
            "error": repr(e),
        })

print("\n생성 완료:", len(created_logs))
print("에러 수:", len(error_logs))
print("case 분포:", case_counter)
print("저장 위치:", AUG_ROOT)

if error_logs:
    print("\n에러 예시:")
    for e in error_logs[:20]:
        print(e)

BACK_AUG 생성 시작: 600장


generate BACK_AUG:   0%|          | 0/600 [00:00<?, ?it/s]


생성 완료: 600
에러 수: 0
case 분포: Counter({'top_front_back': 212, 'middle_front_empty_back': 150, 'top_back_only_lane': 148, 'dense_back_row': 90})
저장 위치: /content/drive/MyDrive/먼작귀/dataset/synthetic_front_back_aug


In [ ]:
# ============================================================
# 6. 생성 결과 검수
# ============================================================

import json
from collections import Counter
from pathlib import Path

case_counter = Counter()

for json_path in AUG_JSON_ROOT.glob("*.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    image_info = data.get("image", {})
    case_type = image_info.get("back_aug_case_type") or image_info.get("sub_scenario_code")

    case_counter[case_type] += 1

print("===== BACK_AUG 유형별 개수 =====")
for case_type, count in case_counter.most_common():
    print(f"{case_type}: {count}")

print("\n총합:", sum(case_counter.values()))

===== BACK_AUG 유형별 개수 =====
top_front_back: 212
middle_front_empty_back: 150
top_back_only_lane: 148
dense_back_row: 90

총합: 600


In [ ]:
# ============================================================
# 7. YOLO txt 형식 검수
# ============================================================

problem_lines = []
class_counter = Counter()
empty_files = []

for txt_path in AUG_YOLO_ROOT.glob("*.txt"):
    lines = [x for x in txt_path.read_text(encoding="utf-8").splitlines() if x.strip()]

    if not lines:
        empty_files.append(str(txt_path))
        continue

    for line_no, line in enumerate(lines, start=1):
        parts = line.split()

        if len(parts) != 5:
            problem_lines.append((str(txt_path), line_no, line, "컬럼 수 오류"))
            continue

        try:
            cid = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:])
        except Exception as e:
            problem_lines.append((str(txt_path), line_no, line, f"파싱 오류: {e}"))
            continue

        if cid < 0 or cid > 66:
            problem_lines.append((str(txt_path), line_no, line, "class_id 범위 오류"))
            continue

        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
            problem_lines.append((str(txt_path), line_no, line, "bbox 범위 오류"))
            continue

        class_counter[cid] += 1

print("문제 line 수:", len(problem_lines))
print("빈 txt 수:", len(empty_files))
print("전체 bbox 수:", sum(class_counter.values()))
print("사용 class 수:", len(class_counter))
print("class_id 최소:", min(class_counter.keys()) if class_counter else None)
print("class_id 최대:", max(class_counter.keys()) if class_counter else None)

if problem_lines:
    print("문제 예시:")
    for x in problem_lines[:20]:
        print(x)
    raise ValueError("YOLO 라벨 형식 문제가 있습니다.")

print("YOLO 라벨 형식 정상")

문제 line 수: 0
빈 txt 수: 0
전체 bbox 수: 62586
사용 class 수: 23
class_id 최소: 10
class_id 최대: 66
YOLO 라벨 형식 정상


In [ ]:
# ============================================================
# 8. JSON 포함 규칙 검수
# ============================================================

json_rule_stats = Counter()
json_rule_errors = []

for json_path in AUG_JSON_ROOT.glob("*.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    image_info = data.get("image", {})

    if image_info.get("scenario_code") != "BACK_AUG":
        json_rule_errors.append((str(json_path), "scenario_code가 BACK_AUG가 아님"))

    objects = data.get("objects", [])

    non_top_front_keys = set()

    for obj in objects:
        row_no = int(obj.get("row_no"))
        depth = obj.get("depth_gt")
        key = (obj.get("slot_id"), obj.get("position_index"))

        if row_no != TOP_ROW_NO and depth == "front":
            non_top_front_keys.add(key)

    for obj in objects:
        row_no = int(obj.get("row_no"))
        depth = obj.get("depth_gt")
        included = bool(obj.get("included_in_product_yolo"))
        key = (obj.get("slot_id"), obj.get("position_index"))

        if depth not in ["front", "back"]:
            continue

        if row_no == TOP_ROW_NO:
            if included:
                json_rule_stats["top_included"] += 1
            else:
                json_rule_errors.append((str(json_path), obj.get("object_id"), "top row object excluded"))

        else:
            if depth == "front":
                if included:
                    json_rule_stats["non_top_front_included"] += 1
                else:
                    json_rule_errors.append((str(json_path), obj.get("object_id"), "non-top front excluded"))

            elif depth == "back":
                if key in non_top_front_keys:
                    if not included:
                        json_rule_stats["non_top_back_with_front_excluded"] += 1
                    else:
                        json_rule_errors.append((str(json_path), obj.get("object_id"), "non-top back with front included"))
                else:
                    if included:
                        json_rule_stats["non_top_back_without_front_included"] += 1
                    else:
                        json_rule_errors.append((str(json_path), obj.get("object_id"), "non-top back without front excluded"))

print("JSON 규칙 통계:")
print(json_rule_stats)

print("JSON 규칙 에러 수:", len(json_rule_errors))

if json_rule_errors:
    print("에러 예시:")
    for x in json_rule_errors[:20]:
        print(x)
    raise ValueError("JSON included_in_product_yolo 규칙 문제가 있습니다.")

print("JSON 포함 규칙 정상")

JSON 규칙 통계:
Counter({'non_top_front_included': 40237, 'non_top_back_with_front_excluded': 37816, 'top_included': 20973, 'non_top_back_without_front_included': 1376})
JSON 규칙 에러 수: 0
JSON 포함 규칙 정상
